# 4.2 Test Design — Titanic Survival Prediction

> **CRISP-DM Phase:** 4. Modeling | **Task:** 4.2 Generate Test Design
>
> This notebook implements and validates the experimental framework for model building: data splitting, evaluation metrics, baseline definition, and experiment tracking configuration.

**Design decisions:**
- **Splitting:** Stratified 5-fold CV on 891 training rows (no separate hold-out — dataset too small)
- **Final evaluation:** Kaggle submission (418 test rows, no local ground truth)
- **Primary metric:** Accuracy (Kaggle's metric)
- **Target:** ~85% accuracy (stretch), minimum 80%
- **Baseline:** Gender-only model (76.5% Kaggle accuracy)

In [1]:
# Setup & path resolution
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
if (PROJECT_ROOT / "notebooks").is_dir():
    pass  # cwd is project root
elif (PROJECT_ROOT.parent / "notebooks").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent  # cwd is a subdirectory

DATA_DIR = PROJECT_ROOT / "data" / "processed"

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, confusion_matrix,
    make_scorer, classification_report
)
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
N_FOLDS = 5

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Random seed: {RANDOM_SEED}")
print(f"CV folds: {N_FOLDS}")

Project root: /Users/tba8ydd/Documents/claude-template
Data directory: /Users/tba8ydd/Documents/claude-template/data/processed
Random seed: 42
CV folds: 5


In [2]:
# Load prepared data
train = pd.read_csv(DATA_DIR / "train_formatted.csv")
test = pd.read_csv(DATA_DIR / "test_formatted.csv")

# Separate features and target
TARGET = "Survived"
ID_COL = "PassengerId"

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET]
X_test = test.drop(columns=[ID_COL])
test_ids = test[ID_COL]

print(f"Training set: {X.shape[0]} rows, {X.shape[1]} features")
print(f"Test set: {X_test.shape[0]} rows, {X_test.shape[1]} features")
print(f"Target distribution: {y.value_counts(normalize=True).to_dict()}")
print(f"Features: {list(X.columns)}")

Training set: 891 rows, 26 features
Test set: 418 rows, 26 features
Target distribution: {0: 0.6161616161616161, 1: 0.3838383838383838}
Features: ['Pclass', 'Sex', 'Age', 'Fare', 'FareLog', 'SibSp', 'Parch', 'FamilySize', 'FamilySizeBin', 'IsAlone', 'HasCabin', 'TicketGroupSize', 'AgeMissing', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mrs']


## 1. Splitting Strategy: Stratified 5-Fold Cross-Validation

**Rationale:**
- **Not time series** — single historical event, so random stratified splits are appropriate
- **No separate validation hold-out** — with only 891 rows, reserving a hold-out would reduce an already small training set. 5-fold CV uses all data for both training and validation.
- **Stratification** preserves the 38.4% survival rate in each fold, preventing class imbalance artifacts
- **Kaggle test set (418 rows)** serves as the final unseen evaluation — ground truth unavailable locally

```
┌─────────────────────────────────────────────────────────┐
│                  891 Training Rows                       │
│  ┌─────────┬─────────┬─────────┬─────────┬─────────┐    │
│  │ Fold 1  │ Fold 2  │ Fold 3  │ Fold 4  │ Fold 5  │    │
│  │ ~178    │ ~178    │ ~178    │ ~178    │ ~179    │    │
│  │ ~38.4%  │ ~38.4%  │ ~38.4%  │ ~38.4%  │ ~38.4%  │    │
│  └─────────┴─────────┴─────────┴─────────┴─────────┘    │
│                                                          │
│  Iteration 1: [VAL]  [TRAIN] [TRAIN] [TRAIN] [TRAIN]    │
│  Iteration 2: [TRAIN] [VAL]  [TRAIN] [TRAIN] [TRAIN]    │
│  Iteration 3: [TRAIN] [TRAIN] [VAL]  [TRAIN] [TRAIN]    │
│  Iteration 4: [TRAIN] [TRAIN] [TRAIN] [VAL]  [TRAIN]    │
│  Iteration 5: [TRAIN] [TRAIN] [TRAIN] [TRAIN] [VAL]     │
└─────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────┐
│           418 Kaggle Test Rows (Final Eval)              │
│           Ground truth unavailable locally               │
│           Used ONLY for final submission                  │
└─────────────────────────────────────────────────────────┘
```

In [3]:
# Define the CV splitter (reused across all experiments)
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# Validate split sizes and target distribution per fold
print(f"{'Fold':<6} {'Train':>6} {'Val':>6} {'Train %surv':>12} {'Val %surv':>10}")
print("-" * 44)

for i, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
    train_surv = y.iloc[train_idx].mean()
    val_surv = y.iloc[val_idx].mean()
    print(f"{i:<6} {len(train_idx):>6} {len(val_idx):>6} {train_surv:>11.1%} {val_surv:>9.1%}")

print(f"\n{'Overall':<6} {'':<6} {'':<6} {y.mean():>11.1%}")

Fold    Train    Val  Train %surv  Val %surv
--------------------------------------------
1         712    179       38.3%     38.5%
2         713    178       38.4%     38.2%
3         713    178       38.4%     38.2%
4         713    178       38.4%     38.2%
5         713    178       38.3%     38.8%

Overall                     38.4%


In [4]:
# Verify no data leakage: check that train and test PassengerIds are disjoint
train_ids_set = set(train[ID_COL])
test_ids_set = set(test[ID_COL])
overlap = train_ids_set & test_ids_set

print(f"Train PassengerId range: {min(train_ids_set)}–{max(train_ids_set)}")
print(f"Test PassengerId range:  {min(test_ids_set)}–{max(test_ids_set)}")
print(f"Overlap: {len(overlap)} passengers")
assert len(overlap) == 0, "DATA LEAKAGE: overlapping PassengerIds!"
print("✓ No overlap — clean split confirmed")

Train PassengerId range: 1–891
Test PassengerId range:  892–1309
Overlap: 0 passengers
✓ No overlap — clean split confirmed


In [5]:
# Compare feature distributions between train and test to detect covariate shift
print("Feature distribution comparison (train vs test):")
print(f"\n{'Feature':<20} {'Train mean':>11} {'Test mean':>11} {'Diff':>8}")
print("-" * 54)

for col in X.columns:
    train_mean = X[col].mean()
    test_mean = X_test[col].mean()
    diff = abs(train_mean - test_mean)
    flag = " ⚠" if diff > 0.1 else ""
    print(f"{col:<20} {train_mean:>11.4f} {test_mean:>11.4f} {diff:>7.4f}{flag}")

Feature distribution comparison (train vs test):

Feature               Train mean   Test mean     Diff
------------------------------------------------------
Pclass                    2.3086      2.2656  0.0431
Sex                       0.3524      0.3636  0.0112
Age                      29.3717     29.7596  0.3879 ⚠
Fare                     32.2042     35.5612  3.3570 ⚠
FareLog                   2.9622      3.0141  0.0519
SibSp                     0.5230      0.4474  0.0756
Parch                     0.3816      0.3923  0.0108
FamilySize                1.9046      1.8397  0.0649
FamilySizeBin             0.4669      0.4426  0.0243
IsAlone                   0.6027      0.6053  0.0026
HasCabin                  0.2290      0.2177  0.0113
TicketGroupSize           2.1212      2.0598  0.0614
AgeMissing                0.1987      0.2057  0.0071
Deck_A                    0.0168      0.0167  0.0001
Deck_B                    0.0527      0.0431  0.0097
Deck_C                    0.0662      0.08

## 2. Evaluation Metrics

| Metric | Role | Threshold | Rationale |
|--------|------|-----------|-----------|
| **Accuracy** | Primary | ≥ 85% CV (target), ≥ 80% (minimum) | Kaggle's sole metric; user's target |
| **F1-score** | Secondary | — | Balances precision/recall for 38.4% minority class |
| **ROC-AUC** | Secondary | — | Discrimination ability, threshold-independent |
| **CV std** | Stability | < 3% | Ensures consistent fold-to-fold performance (from 1.3) |
| **Precision / Recall** | Diagnostic | — | Understand error types |

**Business translation:** Each 1% accuracy gain ≈ 4 more correct predictions out of 418 test passengers. Reaching 85% means ~355 correct vs ~320 for the gender baseline.

In [6]:
# Define reusable scoring dictionary for cross_validate
scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
}

# Success thresholds
THRESHOLDS = {
    "accuracy_target": 0.85,   # user's preferred target
    "accuracy_minimum": 0.80,  # minimum acceptable (from 1.3, updated)
    "accuracy_baseline": 0.765,  # gender-only Kaggle score
    "cv_std_max": 0.03,        # max acceptable CV std (from 1.3)
    "min_improvement_over_baseline": 0.035,  # ~3.5% over baseline to justify complexity
}

print("Scoring metrics:", list(scoring.keys()))
print("\nSuccess thresholds:")
for k, v in THRESHOLDS.items():
    print(f"  {k}: {v}")

Scoring metrics: ['accuracy', 'f1', 'roc_auc', 'precision', 'recall']

Success thresholds:
  accuracy_target: 0.85
  accuracy_minimum: 0.8
  accuracy_baseline: 0.765
  cv_std_max: 0.03
  min_improvement_over_baseline: 0.035


In [7]:
def evaluate_model_cv(model, X, y, cv, scoring, model_name="Model"):
    """Run cross-validation and return a summary DataFrame.
    
    Reusable helper for task 4.3 (Build Model). Fits the model using
    cross_validate, computes mean and std for each metric, and checks
    against success thresholds.
    """
    results = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=False)
    
    summary = {}
    for metric in scoring:
        scores = results[f"test_{metric}"]
        summary[metric] = {
            "mean": scores.mean(),
            "std": scores.std(),
            "per_fold": scores.tolist(),
        }
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"  {model_name} — {cv.n_splits}-Fold Stratified CV Results")
    print(f"{'='*60}")
    print(f"{'Metric':<12} {'Mean':>8} {'± Std':>8} {'Folds':>40}")
    print("-" * 70)
    for metric, vals in summary.items():
        folds_str = " ".join(f"{v:.3f}" for v in vals["per_fold"])
        print(f"{metric:<12} {vals['mean']:>8.4f} {vals['std']:>7.4f}  [{folds_str}]")
    
    # Check against thresholds
    acc_mean = summary["accuracy"]["mean"]
    acc_std = summary["accuracy"]["std"]
    print(f"\n--- Threshold checks ---")
    print(f"  Accuracy ≥ {THRESHOLDS['accuracy_target']:.0%} (target):  {'PASS' if acc_mean >= THRESHOLDS['accuracy_target'] else 'FAIL'} ({acc_mean:.1%})")
    print(f"  Accuracy ≥ {THRESHOLDS['accuracy_minimum']:.0%} (minimum): {'PASS' if acc_mean >= THRESHOLDS['accuracy_minimum'] else 'FAIL'} ({acc_mean:.1%})")
    print(f"  Accuracy > baseline ({THRESHOLDS['accuracy_baseline']:.1%}): {'PASS' if acc_mean > THRESHOLDS['accuracy_baseline'] else 'FAIL'} ({acc_mean:.1%})")
    print(f"  CV std < {THRESHOLDS['cv_std_max']:.0%}:             {'PASS' if acc_std < THRESHOLDS['cv_std_max'] else 'FAIL'} ({acc_std:.1%})")
    
    return summary

print("evaluate_model_cv() defined — ready for use in 4.3")

evaluate_model_cv() defined — ready for use in 4.3


## 3. Baseline: Gender-Only Model

The gender-only baseline predicts all females survive and all males perish. It achieves 76.5% on Kaggle. We implement it as a `DummyClassifier` using the `Sex` column to validate our CV setup matches the known baseline performance.

In [8]:
# Gender-only baseline: predict Survived = Sex (where Sex=1 is female)
# This directly uses the Sex column as the prediction
from sklearn.base import BaseEstimator, ClassifierMixin

class GenderBaseline(BaseEstimator, ClassifierMixin):
    """Predicts survival based on Sex column only (female=1 → survived=1)."""
    
    def fit(self, X, y=None):
        self.classes_ = np.array([0, 1])
        return self
    
    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            return X["Sex"].values.astype(int)
        # If numpy array, Sex is the column at index that matches
        return X[:, list(range(X.shape[1]))[0]].astype(int)
    
    def predict_proba(self, X):
        preds = self.predict(X)
        return np.column_stack([1 - preds, preds])

# Run CV on the gender baseline
baseline_summary = evaluate_model_cv(
    GenderBaseline(), X, y, cv, scoring, model_name="Gender Baseline"
)

# The gender baseline on training data should be ~78.7% (not 76.5% — that's Kaggle test)
print(f"\nNote: 76.5% is the Kaggle test accuracy. CV accuracy on training data may differ slightly.")


  Gender Baseline — 5-Fold Stratified CV Results
Metric           Mean    ± Std                                    Folds
----------------------------------------------------------------------
accuracy       0.7868  0.0188  [0.788 0.775 0.787 0.764 0.820]
f1             0.7094  0.0316  [0.716 0.683 0.689 0.691 0.768]
roc_auc           nan     nan  [nan nan nan nan nan]
precision      0.7434  0.0302  [0.738 0.741 0.778 0.691 0.768]
recall         0.6810  0.0535  [0.696 0.632 0.618 0.691 0.768]

--- Threshold checks ---
  Accuracy ≥ 85% (target):  FAIL (78.7%)
  Accuracy ≥ 80% (minimum): FAIL (78.7%)
  Accuracy > baseline (76.5%): PASS (78.7%)
  CV std < 3%:             PASS (1.9%)

Note: 76.5% is the Kaggle test accuracy. CV accuracy on training data may differ slightly.


## 4. Data Leakage Prevention

| Risk | Mitigation | Status |
|------|-----------|--------|
| Preprocessing leakage | All imputers/encoders fit on train only in tasks 3.2–3.5 | ✓ Handled |
| CV-internal leakage | StandardScaler fit inside each CV fold via `Pipeline` (for LR, SVM) | ✓ Design in place |
| Group leakage | N/A — each passenger is unique, no repeated entities | ✓ Not applicable |
| Temporal leakage | N/A — single historical event, no temporal dimension | ✓ Not applicable |
| Target leakage | All features are pre-event passenger attributes; no post-event information | ✓ Verified in 2.3 |
| Train/test overlap | PassengerId ranges disjoint (1–891 vs 892–1309) | ✓ Verified above |

## 5. MLflow Experiment Tracking Plan

| Category | Details |
|----------|---------|
| **Experiment name** | `titanic-survival-modeling` |
| **Run naming** | `{technique}-{variant}-{YYYYMMDD}` (e.g., `xgboost-tuned-20260331`) |
| **Parameters** | Hyperparameters, feature set version, split config (seed, n_folds), data version (DVC hash) |
| **Metrics** | Per-fold and mean±std for: accuracy, f1, roc_auc, precision, recall |
| **Artifacts** | Trained model, feature importance plot, confusion matrix, classification report |
| **Tags** | `technique_family`, `phase` (baseline/default/tuned/final), `data_version`, `author` |

### Experiment Comparison Rules
- Models compared on **CV validation metrics** during development
- Kaggle test set used **only for final selected model(s)** (max 10 submissions/day)
- All comparisons include the gender baseline as reference
- A model must exceed baseline by ≥ 3.5% CV accuracy to justify its complexity

## 6. Reproducibility Checklist

- [x] Random seed: **42** — used in `StratifiedKFold` and all model initializations
- [ ] Data version: tracked via DVC hash (to be logged in MLflow)
- [ ] Code version: tracked via git commit SHA (to be logged in MLflow)
- [ ] Environment: tracked via `requirements.txt`
- [x] Split logic: deterministic given seed and `StratifiedKFold(shuffle=True, random_state=42)`
- [x] CV splitter object (`cv`) reusable across all experiments in 4.3